<a href="https://colab.research.google.com/github/DonChenn/RedditSentimentAnalysis/blob/main/BERTopicModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

First we import our libraries and We download the Dataset

In [1]:
!pip install bertopic
!pip install gensim
from google.colab import drive
import sys
import kagglehub
from bertopic import BERTopic
import pandas as pd
import os
path = kagglehub.dataset_download("neelgajare/liberals-vs-conservatives-on-reddit-13000-posts")
csv_file = None
for filename in os.listdir(path):
    if filename.endswith(".csv"):
        csv_file = os.path.join(path, filename)
        break

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 63.7 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


100%|██████████| 2.09M/2.09M [00:00<00:00, 115MB/s]

Extracting files...


Once dataset is loaded in we replace reddit posts with titles but no text with empty strings. We also create a new column that cobimes the title and the text columns to allow the model to have more information to predict the topic.

In [2]:
if csv_file:
    df = pd.read_csv(csv_file)
    df['Title'] = df['Title'].fillna('')
    df['Text'] = df['Text'].fillna('')
    df['Combined_Content'] = df['Title'] + " " + df['Text']

else:
    print("Error: No CSV file found in the downloaded folder.")

Then we tokenize our combined content columns removing stopwords and non alphabetic chars.We initialize the BERTopic model and then run it on the cleaned combined content column.

In [3]:

docs = df['Combined_Content'].tolist()
topic_model = BERTopic(language="english", calculate_probabilities=True, verbose=True, nr_topics=50)
topics, probs = topic_model.fit_transform(docs)
freq = topic_model.get_topic_info()

2026-02-17 23:40:22,159 - BERTopic - Embedding - Transforming documents to embeddings.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/402 [00:00<?, ?it/s]

2026-02-17 23:45:48,453 - BERTopic - Embedding - Completed ✓
2026-02-17 23:45:48,457 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-17 23:46:26,019 - BERTopic - Dimensionality - Completed ✓
2026-02-17 23:46:26,021 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-17 23:46:57,453 - BERTopic - Cluster - Completed ✓
2026-02-17 23:46:57,454 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-02-17 23:46:58,271 - BERTopic - Representation - Completed ✓
2026-02-17 23:46:58,272 - BERTopic - Topic reduction - Reducing number of topics
2026-02-17 23:46:58,310 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-17 23:46:59,079 - BERTopic - Representation - Completed ✓
2026-02-17 23:46:59,084 - BERTopic - Topic reduction - Reduced number of topics from 188 to 50


In [17]:
topic_info = topic_model.get_topic_info()
raw_labels = topic_info['Name'].tolist()
formatted_labels = []

for label in raw_labels:
    parts = label.split('_')
    if parts[0] == "-1":
        clean_label = "Other / Irrelevant / Noise"
    else:
        clean_label = ", ".join(parts[1:])
    formatted_labels.append(clean_label)

for i, label in enumerate(formatted_labels):
    print(f"Index {i}: {label}")

llm_topic_list = formatted_labels

Index 0: Other / Irrelevant / Noise
Index 1: ukraine, russia, russian, putin
Index 2: trump, desantis, election, to
Index 3: the, of, and, to
Index 4: women, her, she, men
Index 5: the, to, police, of
Index 6: covid, vaccine, insurance, healthcare
Index 7: podcast, this, meme, you
Index 8: the, of, and, war
Index 9: democracy, you, to, social
Index 10: workers, wage, minimum, unions
Index 11: tax, debt, student, poverty
Index 12: biden, bidens, joe, poll
Index 13: cuba, venezuela, cuban, us
Index 14: inflation, bitcoin, crypto, fed
Index 15: facebook, truth, social, app
Index 16: china, chinese, north, korea
Index 17: climate, change, carbon, woke
Index 18: abortion, texas, law, roe
Index 19: land, property, housing, rent
Index 20: tucker, manchin, carlson, joe
Index 21: elon, musk, tesla, musks
Index 22: the, party, catalan, and
Index 23: labour, kshama, sawant, recall
Index 24: market, monopolies, public, free
Index 25: mask, masks, mandates, mandate
Index 26: schools, education, boo